# Appending grades from .JSONL to df

In [ ]:
# ==== IMPORTS ====
import pandas as pd
import json


# ==== SETTINGS ====
# ---- IMPORT SETTINGS ----
IMPORT_PATH = "../data/gcp_manual_copy/"

# ---- EXPORT SETTINGS ----
EXPORT = True
EXPORT_PATH = IMPORT_PATH

# ---- FILES ----
# Metrics files
FILE_ALL = "thesis_meta_all_metrics_except_grade_27032026.parquet"
FILE_FILTERED = "thesis_meta_all_metrics_except_grade_filtered_27032026.parquet"

# Grade file
JSONL_NAME = "dtu_findit-extraction_and_processing-grading_results.jsonl"

## Loading df dataset

In [ ]:
# ==== FUNCTION ====
def load_csv_to_df(csv_path, sep=";"):
    try:
        df = pd.read_csv(csv_path, encoding="utf-8", sep=sep)
        print(f"Successfully loaded CSV from {csv_path}")
        print(f"DataFrame shape: {df.shape}")
        print(f"DataFrame columns: {df.columns.tolist()}\n")
        return df
    except Exception as e:
        print(f"Error loading CSV from {csv_path}: {e}")
        return None

def load_parquet_to_df(parquet_path, na=False):
    try:
        df = pd.read_parquet(parquet_path)
        print(f"Successfully loaded Parquet from {parquet_path}")
        print(f"DataFrame shape: {df.shape}")
        if na:
            print(f"DataFrame N/A counts:\n{df.isna().sum()}\n")
        print(f"DataFrame columns: {df.columns.tolist()}\n")
        return df
    except Exception as e:
        print(f"Error loading Parquet from {parquet_path}: {e}")
        return None

# ==== LOAD DATAFRAMES ====
df_all = load_parquet_to_df(IMPORT_PATH + FILE_ALL)
df_filtered = load_parquet_to_df(IMPORT_PATH + FILE_FILTERED)

# ==== DROP NA COLUMNS ====
df_all_noNA = df_all.dropna(axis=1, how="all")
print(f"df_all_noNA shape: {df_all_noNA.shape}")
print(f"df_all_noNA columns: {df_all_noNA.columns.tolist()}\n")
df_filtered_noNA = df_filtered.dropna(axis=1, how="all")
print(f"df_filtered_noNA shape: {df_filtered_noNA.shape}")
print(f"df_filtered_noNA columns: {df_filtered_noNA.columns.tolist()}\n")

# ==== COLUMNS TO DROP ====
drop_columns = [
    "access_ss",
    "Affiliations",
    "collection_facet",
    "format",
    "fulltext_availability_facet",
    "ISBN",
    "Journal Page",
    "isolanguage_facet",
    "Publisher",
    "Source",
    "source_all_ss",
    "match_trigger",
    "equation_pipeline_version",
    "pdf_file_analysis",
    "num_tot_pages_analysis",
    "num_cont_pages_analysis",
    "num_words_full_analysis",
    "num_words_cont_analysis",
    "abstract_ts_analysis",
    "Author_analysis",
    "Publication Year_analysis",
    "primary_member_id_s_analysis",
    "Title_analysis",
    ]

# ==== DROP COLUMNS ====
df_all_noNA_clean = df_all_noNA.drop(columns=drop_columns, errors="ignore")
print(f"df_all_noNA_clean shape: {df_all_noNA_clean.shape}")
print(f"df_all_noNA_clean columns: {df_all_noNA_clean.columns.tolist()}\n")
df_filtered_noNA_clean = df_filtered_noNA.drop(columns=drop_columns, errors="ignore")
print(f"df_filtered_noNA_clean shape: {df_filtered_noNA_clean.shape}")
print(f"df_filtered_noNA_clean columns: {df_filtered_noNA_clean.columns.tolist()}\n")

# ==== LOCK DATAFRAMES FOR ANALYSIS ====
print("Final DataFrames for Analysis:")
df_all_final = df_all_noNA_clean.copy()
print(f"df_all_final shape: {df_all_final.shape}")
df_filtered_final = df_filtered_noNA_clean.copy()
print(f"df_filtered_final shape: {df_filtered_final.shape}")

## Loading .JSONL

In [ ]:
# ==== LOAD .JSONL ====
# Load the lines manually to handle the nested structure effectively
data = []
with open(IMPORT_PATH + JSONL_NAME, 'r') as f:
    for line in f:
        data.append(json.loads(line))

# Flatten the 'data' and 'meta' keys into the main dataframe
df_grades = pd.json_normalize(data)

# Resulting columns will look like: 'data.scientific_contribution', 'meta.attempts', etc.
print(df_grades.shape)
display(df_grades.head())

In [ ]:
# ==== PREPARE GRADE COLUMNS FOR LEFT JOIN ====
grade_column_map = {
    "data.scientific_contribution": "grading_scientific_contribution",
    "data.methodological_rigor": "grading_methodological_rigor",
    "data.technical_implementation": "grading_technical_implementation",
    "data.literature_review": "grading_literature_review",
    "data.process_professionalism": "grading_process_professionalism",
    "data.impact_applicability": "grading_impact_applicability",
    "data.research_question_alignment": "grading_research_question_alignment",
    "data.total_score": "grading_total_score",
    "meta.attempts": "grading_meta_attempts",
    "meta.original_chars": "grading_meta_original_chars",
    "meta.trimmed_at_references": "grading_meta_trimmed_at_references",
    "meta.input_chars": "grading_meta_input_chars",
    "meta.estimated_input_tokens": "grading_meta_estimated_input_tokens",
    "meta.was_truncated": "grading_meta_was_truncated",
    "meta.prompt_fit_attempts": "grading_meta_prompt_fit_attempts",
    "meta.timeout_seconds": "grading_meta_timeout_seconds",
    "meta.stream_mode": "grading_meta_stream_mode",
}

required_grade_cols = ["file"] + list(grade_column_map.keys())
missing_in_grades = [c for c in required_grade_cols if c not in df_grades.columns]
if missing_in_grades:
    raise KeyError(f"Missing required columns in df_grades: {missing_in_grades}")

# Keep only relevant columns and rename to target grading_* names
df_grades_for_join = df_grades[required_grade_cols].rename(columns=grade_column_map).copy()

# Normalize join keys to reduce mismatch risk from whitespace
df_grades_for_join["file"] = df_grades_for_join["file"].astype(str).str.strip()
df_filtered_final["pdf_file"] = df_filtered_final["pdf_file"].astype(str).str.strip()

# Detect duplicate grade keys and keep the row with the most non-null grade/meta values
target_grade_cols = list(grade_column_map.values())
dup_count = df_grades_for_join["file"].duplicated().sum()
if dup_count > 0:
    df_grades_for_join["_grade_non_null_count"] = df_grades_for_join[target_grade_cols].notna().sum(axis=1)
    df_grades_for_join = (
        df_grades_for_join
        .sort_values(["file", "_grade_non_null_count"], ascending=[True, False])
        .drop_duplicates(subset=["file"], keep="first")
        .drop(columns=["_grade_non_null_count"])
    )
    print(
        f"Duplicate file keys found in df_grades: {dup_count}. "
        "Kept the most complete row per file key (highest non-null grade/meta count)."
    )

# If target grading columns already exist on the left, drop them before merge to avoid _x/_y suffixes
existing_target_cols = [c for c in target_grade_cols if c in df_filtered_final.columns]
if existing_target_cols:
    print(f"Dropping existing grading columns before merge: {existing_target_cols}")
    df_filtered_base = df_filtered_final.drop(columns=existing_target_cols).copy()
else:
    df_filtered_base = df_filtered_final.copy()

# ==== LEFT JOIN ENRICHMENT ====
df_filtered_final_enriched = df_filtered_base.merge(
    df_grades_for_join,
    how="left",
    left_on="pdf_file",
    right_on="file",
    validate="m:1",
).drop(columns=["file"] )

print(f"df_filtered_final_enriched shape: {df_filtered_final_enriched.shape}. Compared to df_filtered_final shape: {df_filtered_final.shape}.")
print("Added grading columns:")
print(target_grade_cols)

# Quick match diagnostics
score_col = "grading_total_score" if "grading_total_score" in df_filtered_final_enriched.columns else None
if score_col is None:
    available = [c for c in df_filtered_final_enriched.columns if "total_score" in c]
    raise KeyError(
        "Could not find grading_total_score after merge. "
        f"Available total_score-like columns: {available}"
    )

matched_rows = df_filtered_final_enriched[score_col].notna().sum()
total_rows = len(df_filtered_final_enriched)
print(f"Matched rows with grades: {matched_rows}/{total_rows} ({matched_rows/total_rows:.1%})")

display(df_filtered_final_enriched.head())

## EXPORTING THE DF

In [ ]:
pct_grades = round(matched_rows/total_rows * 100)
date_stamp = pd.Timestamp.now().strftime("%d%m%Y")

# ==== EXPORT ENRICHED DATAFRAME ====
if EXPORT:
    export_name = f"thesis_meta_all_metrics_with_{pct_grades}pct_grades_{date_stamp}.parquet"
    df_filtered_final_enriched.to_parquet(EXPORT_PATH + export_name, index=False)
    print(f"Exported enriched DataFrame to {EXPORT_PATH + export_name}")
else:
    print("Enriched DataFrame not exported.")